# XVAMP Quick Start

This notebook will compute and plot the refraction and absorption profile from the
default [xvamp.model.Duan2010](../xvamp/model.rst#xvamp.model.Duan2010) model. The first
time XVAMP is loaded, everything might take longer, since all datasets are preloaded.

> **Note:** The model is still under development, so the values shown here might still
> change. Also, there are a couple of options that can modify the default
> [xvamp.model.Duan2010](../xvamp/model.rst#xvamp.model.Duan2010) behavior, which are
> documented in the API, but not shown here.

In [ ]:
# some basic imports
import numpy as np
import matplotlib.pyplot as plt
from cmcrameri import cm
from xvamp.utils import geometric_range_from_central_angle
from xvamp.model import Duan2010
from astropy.units import Quantity
from astropy.visualization import quantity_support
quantity_support()
%config InlineBackend.figure_formats = ["svg", "pdf"]

Now, let's instantiate the model with all default settings:

In [ ]:
model = Duan2010()

We're essentially all done. The `model` object now contains a variety of attributes,
most importantly the different temperature, pressure, compositional, etc. profiles:

In [ ]:
list(model.__dict__.keys())

The description, as well as a list of methods, can be found in the API under
[xvamp.model.Model](../xvamp/model.rst#xvamp.model.Model). We will plot a couple of
them here.

## Temperature and pressure

While plotting, we make nice use of the fact that the `model` uses astropy
[Quantity](https://docs.astropy.org/en/stable/units/quantity.html)s.

In [ ]:
fig, axes = plt.subplots(ncols=2, sharey=True, figsize=(8, 4), layout="constrained")
axes[0].plot(model.temperature, model.altitude)
axes[0].set_title("Temperature")
axes[1].plot(model.pressure.to("bar"), model.altitude)  # automatically sets axes units!
axes[1].set_title("Pressure")
axes[1].set_xscale("log")
axes[1].set_xlim(1e-20, 1e3)
axes[1].set_ylim(-10, 300)
for ax in axes:
    ax.grid()

## Species molar fraction, polarization, absorption, and refraction

Let's now plot an overview of the species present in the atmosphere, as well as their
contribution to the overall polarization and absorption. Let's also plot the joint
absorption and polarization, and the final refraction profile.

In [ ]:
fig, axes = plt.subplots(
    ncols=4, sharey=True, figsize=(12, 5), width_ratios=[5, 5, 5, 3], dpi=300
)
axes[2].semilogx(
    model.absorptions["CO2+N2+AR+H2O"].to("dB/km"),
    model.altitude,
    c=f"C8",
    label="CO2+N2+AR+H2O",
    zorder=2,
)
for i, comp in enumerate(model.molar_fractions.colnames):
    axes[0].semilogx(
        model.molar_fractions[comp], model.altitude, c=f"C{i}", label=comp, zorder=2
    )
    if comp in model.polarizations.colnames:
        axes[1].semilogx(
            model.polarizations[comp] * model.molar_fractions[comp],
            model.altitude,
            c=f"C{i}",
            label=comp,
            zorder=2,
        )
    if comp in model.absorptions.colnames:
        axes[2].semilogx(
            model.absorptions[comp].to("dB/km"),
            model.altitude,
            c=f"C{i}",
            label=comp,
            zorder=2,
        )
axes[1].semilogx(
    model.polarizations["cloud"],
    model.altitude,
    c=f"C9",
    ls="--",
    zorder=2,
    label="Cloud",
)
axes[1].semilogx(
    model.polarization, model.altitude, c=f"C7", lw=4, ls="--", zorder=1, label="Total"
)
axes[2].semilogx(
    model.absorptions["cloud"],
    model.altitude,
    c=f"C9",
    ls="--",
    zorder=2,
    label="Cloud",
)
axes[2].semilogx(
    model.absorption, model.altitude, c=f"C7", lw=4, ls="--", zorder=1, label="Total"
)
axes[3].plot(
    model.refraction, model.altitude, c=f"C7", lw=4, ls="--", zorder=1, label="Total"
)
axes[0].set_xlim(1e-2, 2e6)
axes[0].set_ylim(-7, 110)
axes[1].set_xlim(1e-10, 1e6)
axes[1].set_xlabel("Polarization [-]")
axes[2].set_xlim(1e-6, 1)
axes[2].set_xlabel("Absorption [dB/km]")
axes[0].set_ylabel("Altitude [km]")
axes[0].set_xlabel("Molar Fraction [ppm]")
axes[0].set_yticks(np.arange(0, 101, 25))
axes[3].set_xlabel("Refraction [-]")
for iax, ax in enumerate(axes):
    ax.grid()
    if iax < 2:
        ax.legend(loc="upper left", ncol=2)
    else:
        ax.legend(loc="upper right", ncol=1)

## Profile-integrated values

We can also compute the total delay (defined as the difference between the apparent
and geometric range) and attenuation (two-way) as the signal travels from the
spacecraft to the surface and back for a given spacecraft altitude, terrain height,
and look angle. The relevant functions are all vectorized enough to allow both
height ranges to be computed at the same time.

In [ ]:
# range of test values
height_terrain = Quantity(np.linspace(-3, 11, num=29), "km")
height_platform = Quantity(np.linspace(180, 260, num=81), "km")
look_angle = Quantity(30, "deg")
# get profile-integrated values
delay, attenuation = model.get_delay_attenuation(
    height_terrain, height_platform, look_angle
)

In [ ]:
fig, axes = plt.subplots(ncols=2, sharey=True, figsize=(10, 1.7))
pc0 = axes[0].pcolormesh(
    height_terrain,
    height_platform,
    delay.to("m").value,
    cmap=cm.navia_r,
    rasterized=True,
)
pc1 = axes[1].pcolormesh(
    height_terrain,
    height_platform,
    attenuation.to("dB").value,
    cmap=cm.bamako_r,
    rasterized=True,
)
fig.colorbar(pc0, ax=axes[0], label="Delay [m]")
fig.colorbar(pc1, ax=axes[1], label="Attenuation [dB]")
axes[0].set_ylabel("Platform height [km]")
axes[0].set_yticks(np.arange(180, 261, 20))
for ax in axes:
    ax.set_xlabel("Terrain height [km]")
    ax.set_xticks(np.arange(-3, 12), minor=True)
    ax.set_xticks(np.arange(-2, 12, 2))

## Example of non-default model parameters

In [ ]:
# new model object
model_new = Duan2010(use_h2so4_from="kolodner")
# get profile-integrated values
delay_new, attenuation_new = model_new.get_delay_attenuation(
    height_terrain, height_platform, look_angle
)

In [ ]:
fig, axes = plt.subplots(ncols=2, sharey=True, figsize=(10, 1.7))
pc0 = axes[0].pcolormesh(
    height_terrain,
    height_platform,
    (delay_new - delay).to("mm").value,
    cmap=cm.batlow,
    rasterized=True,
)
pc1 = axes[1].pcolormesh(
    height_terrain,
    height_platform,
    (attenuation_new - attenuation).to("dB").value,
    cmap=cm.lipari,
    rasterized=True,
)
fig.colorbar(pc0, ax=axes[0], label="ΔDelay [mm]")
fig.colorbar(pc1, ax=axes[1], label="ΔAttenuation [dB]")
axes[0].set_ylabel("Platform height [km]")
axes[0].set_yticks(np.arange(180, 261, 20))
for ax in axes:
    ax.set_xlabel("Terrain height [km]")
    ax.set_xticks(np.arange(-3, 12), minor=True)
    ax.set_xticks(np.arange(-2, 12, 2))

In [ ]:
# create figure
fig, ax = plt.subplots()
# add H2SO4 mixing ratios for both models
ax.plot(model.molar_fractions["H2SO4"], model.altitude, label="Default (reprocessed)")
ax.plot(
    model_new.molar_fractions["H2SO4"], model_new.altitude, label="Orbit 3214 (old)"
)
# make pretty
ax.set_xlabel("H2SO4 Mixing Ratio [ppm]")
ax.set_ylabel("Altitude [km]")
ax.set_ylim(30, 60)
ax.legend()

While the default dataset contains less H2SO4, therefore less attenuation, and
therefore a more favorable case for the VISAR instrument operations, it is chosen
as the default since it is a newer, reprocessed dataset, and we can therefore
expect it to be more accurate. The other profiles are included as options in case
we want to challenge the different input assumptions. Other options are available
and documented in [xvamp.model.Duan2010](../xvamp/model.rst#xvamp.model.Duan2010).